# Ejemplo o caso práctico de aplicación de arquitectura lambda con PySpark

Bienvenido a este cuaderno interactivo diseñado desde la perspectiva de la ingeniería de datos en entornos de Big Data. A lo largo de esta guía, construiremos desde cero un flujo de datos completo implementando el patrón de **Arquitectura Lambda** utilizando **Apache Spark (PySpark)**.

---

## 🛠️ Guía paso a paso: Cómo importar y ejecutar este cuaderno (`.ipynb`)

Este laboratorio está diseñado para ejecutarse de principio a fin en **Databricks Free / Community Edition** utilizando el sistema de archivos temporal (`/tmp/`) y fuentes de streaming en memoria/JSON sin requerir servidores externos.

### Opción A: Ejecutar en Databricks Free / Community Edition
1. Inicia sesión en tu cuenta de **Databricks**.
2. En el menú lateral izquierdo, haz clic en **Workspace** (Área de trabajo).
3. Navega a tu carpeta de usuario, haz **clic derecho** en un espacio vacío y selecciona **Import** (Importar).
4. Arrastra o selecciona este archivo `arquitectura_lambda_pyspark-v2.ipynb` desde tu ordenador y haz clic en **Import**.
5. En la esquina superior derecha del cuaderno, asegúrate de conectar tu **Cluster** (Compute) activo y ejecuta las celdas en orden usando `Shift + Enter`.

### Opción B: Alternativa gratuita recomendada (Google Colab)
Si tienes problemas para crear una cuenta en Databricks Community Edition o el clúster gratuito tarda demasiado en arrancar:
1. Entra en [Google Colab](https://colab.research.google.com/) con tu cuenta de Google.
2. En la ventana inicial (o desde el menú **Archivo > Subir cuaderno**), sube este mismo archivo `.ipynb`.
3. Ejecuta las celdas normalmente: **la primera celda de código detectará automáticamente que no estás en Databricks e instalará e inicializará PySpark por su cuenta** sin que tengas que cambiar ni una sola línea de código.

---

## 1. ¿Qué es la Arquitectura Lambda?

En el mundo del Big Data, a menudo nos enfrentamos a un dilema clásico: necesitamos procesar enormes volúmenes de datos históricos con gran precisión (lo cual toma tiempo), pero el negocio también exige ver qué está ocurriendo en este preciso instante con una latencia mínima.

La **Arquitectura Lambda**, propuesta por Nathan Marz, resuelve este problema dividiendo el flujo de datos en tres capas complementarias:

1. **Capa Batch (Procesamiento por Lotes):**
   * **Objetivo:** Precisión y procesamiento de grandes volúmenes históricos.
   * **Funcionamiento:** Almacena el conjunto de datos maestro (*Master Dataset*) de forma inmutable (solo se agregan datos, no se sobrescriben ni eliminan). Periódicamente (por ejemplo, cada noche o cada hora), calcula vistas preagregadas (*Batch Views*) leyendo todo el historial.
   * **Ventaja:** Si hay un error en el código o en los datos, simplemente se corrige y se recalcula todo desde el inicio.

2. **Capa Speed (Procesamiento en Tiempo Real / Velocidad):**
   * **Objetivo:** Baja latencia (milisegundos o segundos).
   * **Funcionamiento:** Procesa únicamente los datos nuevos que han llegado desde la última vez que se ejecutó la Capa Batch. Sacrifica a veces la exactitud compleja en favor de la rapidez, generando vistas en tiempo real (*Realtime Views*).

3. **Capa Serving (Capa de Servicio o Consulta):**
   * **Objetivo:** Unificar la vista para el usuario final o dashboard.
   * **Funcionamiento:** Cuando un analista o sistema hace una consulta, esta capa combina (hace un *merge* o *union*) de los resultados de la **Capa Batch** (datos históricos consolidados) con los de la **Capa Speed** (datos de los últimos minutos), ofreciendo una respuesta completa y actualizada al instante.

---
## Paso 0: Preparación del Entorno y Sesión de Spark

En esta primera celda de código verificaremos en qué entorno estamos trabajando. 
* Si ejecutas esto en **Databricks**, el objeto `spark` ya existe automáticamente.
* Si decides ejecutarlo en **Google Colab** o en tu **computadora local**, el script instalará e inicializará `pyspark` automáticamente.

In [0]:
import os
import time
import shutil

# Verificamos si ya existe una sesión de Spark activa (como ocurre en Databricks)
try:
    spark
    print("✅ Entorno Databricks detectado. La sesión 'spark' ya está activa.")
except NameError:
    print("⚙️ No se detectó sesión previa de Spark. Configurando PySpark (Modo Google Colab / Local)...")
    # Si estamos en Colab y no está instalado pyspark, lo instalamos silenciosamente
    try:
        import pyspark
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "pyspark"], check=True)
    
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName("LambdaArchitecture_PySpark") \
        .master("local[*]") \
        .config("spark.sql.shuffle.partitions", "4") \
        .getOrCreate()
    print("✅ Sesión de Spark creada con éxito.")

# Importamos las funciones clave que usaremos en nuestro proyecto
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType

# Definimos las rutas base donde guardaremos nuestros datos de prueba
BASE_PATH = "/tmp/lambda_ecommerce_lab"
RAW_BATCH_PATH = f"{BASE_PATH}/master_dataset_batch"
BATCH_VIEW_PATH = f"{BASE_PATH}/batch_views"
STREAM_INPUT_PATH = f"{BASE_PATH}/stream_landing_zone"
STREAM_CHECKPOINT_PATH = f"{BASE_PATH}/stream_checkpoint"

# Limpiamos el directorio por si ejecutaste el laboratorio anteriormente (idempotencia)
if os.path.exists(BASE_PATH):
    shutil.rmtree(BASE_PATH)

os.makedirs(RAW_BATCH_PATH, exist_ok=True)
os.makedirs(BATCH_VIEW_PATH, exist_ok=True)
os.makedirs(STREAM_INPUT_PATH, exist_ok=True)

print(f"📁 Directorios de trabajo inicializados en: {BASE_PATH}")

✅ Entorno Databricks detectado. La sesión 'spark' ya está activa.
📁 Directorios de trabajo inicializados en: /tmp/lambda_ecommerce_lab


---
## Caso de Uso: Analítica de Ventas de E-Commerce ("TechStore")

Imagina que eres el Ingeniero de Datos de **TechStore**, una tienda online de tecnología que vende en 4 categorías: `Laptops`, `Smartphones`, `Audio` y `Monitores`.

El equipo directivo necesita un **Dashboard de Ingresos Totales por Categoría**:
1. Tienen millones de registros históricos consolidados hasta el cierre de ayer (**Capa Batch**).
2. Durante el día de hoy, los clientes siguen comprando segundo a segundo (**Capa Speed**).
3. El director quiere consultar en cualquier momento cuál es el **ingreso total acumulado (Ayer + Hoy en vivo)** y el número de transacciones (**Capa Serving**).

---
## Paso 1: Capa Batch (El *Master Dataset* y las *Batch Views*)

### Teoría rápida:
En la **Capa Batch**, los datos brutos llegan a nuestro *Data Lake* (por ejemplo, archivos Parquet en Amazon S3, Azure Data Lake o DBFS en Databricks). Estos datos son **inmutables**. Un proceso programado (Batch Job) lee todo el histórico y calcula las agregaciones pesadas.

Vamos a simular el histórico de ventas consolidadas hasta ayer y a ejecutar nuestro trabajo por lotes para generar la **Vista Batch (`batch_view`)**.

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS banking_lab_amolina.default.lambda_ecommerce_lab")
BASE_PATH = "/Volumes/banking_lab_amolina/default/lambda_ecommerce_lab"
RAW_BATCH_PATH = f"{BASE_PATH}/master_dataset_batch"
BATCH_VIEW_PATH = f"{BASE_PATH}/batch_views"
STREAM_INPUT_PATH = f"{BASE_PATH}/stream_landing_zone"
STREAM_CHECKPOINT_PATH = f"{BASE_PATH}/stream_checkpoint"
os.makedirs(STREAM_INPUT_PATH, exist_ok=True)

# 1. Definimos el esquema estricto de nuestras transacciones (Buena práctica en Ingeniería de Datos)
esquema_ventas = StructType([
    StructField("id_transaccion", StringType(), False),
    StructField("categoria", StringType(), False),
    StructField("monto", DoubleType(), False),
    StructField("fecha_hora", StringType(), False)
])

# 2. Simulamos datos históricos consolidados (Master Dataset hasta el corte de ayer)
datos_historicos = [
    ("TX_001", "Laptops", 1200.00, "2026-05-01 10:15:00"),
    ("TX_002", "Smartphones", 800.00, "2026-05-01 11:20:00"),
    ("TX_003", "Audio", 150.00, "2026-05-01 12:05:00"),
    ("TX_004", "Laptops", 1500.00, "2026-05-01 14:30:00"),
    ("TX_005", "Monitores", 400.00, "2026-05-01 15:45:00"),
    ("TX_006", "Smartphones", 950.00, "2026-05-01 16:10:00"),
    ("TX_007", "Audio", 200.00, "2026-05-01 17:50:00"),
    ("TX_008", "Monitores", 350.00, "2026-05-01 18:20:00"),
    ("TX_009", "Laptops", 1100.00, "2026-05-01 19:00:00"),
    ("TX_010", "Smartphones", 750.00, "2026-05-01 20:15:00")
]

# Creamos el DataFrame y lo guardamos en formato Parquet en nuestro Master Dataset
df_master_raw = spark.createDataFrame(datos_historicos, schema=esquema_ventas) \
    .withColumn("fecha_hora", F.to_timestamp("fecha_hora"))

df_master_raw.write.mode("append").parquet(RAW_BATCH_PATH)
print("✅ 1. Datos históricos guardados en el Master Dataset (Formato Parquet).")

# 3. PROCESO BATCH: Leemos todo el Master Dataset y calculamos la Batch View
df_lectura_batch = spark.read.parquet(RAW_BATCH_PATH)

df_batch_view = df_lectura_batch.groupBy("categoria").agg(
    F.count("id_transaccion").alias("total_transacciones_batch"),
    F.sum("monto").alias("ingresos_batch"),
    F.max("fecha_hora").alias("ultima_actualizacion_batch")
)

# Guardamos la vista precalculada para que la Capa Serving pueda leerla rápido
df_batch_view.write.mode("overwrite").parquet(BATCH_VIEW_PATH)

print("📊 2. Vista Batch (Batch View) generada con éxito:")
df_batch_view.orderBy(F.desc("ingresos_batch")).show(truncate=False)

✅ 1. Datos históricos guardados en el Master Dataset (Formato Parquet).
📊 2. Vista Batch (Batch View) generada con éxito:
+-----------+-------------------------+--------------+--------------------------+
|categoria  |total_transacciones_batch|ingresos_batch|ultima_actualizacion_batch|
+-----------+-------------------------+--------------+--------------------------+
|Laptops    |3                        |3800.0        |2026-05-01 19:00:00       |
|Smartphones|3                        |2500.0        |2026-05-01 20:15:00       |
|Monitores  |2                        |750.0         |2026-05-01 18:20:00       |
|Audio      |2                        |350.0         |2026-05-01 17:50:00       |
+-----------+-------------------------+--------------+--------------------------+



---
## Paso 2: Capa Speed (Procesamiento en Tiempo Real con Spark Structured Streaming)

### Teoría rápida:
Mientras nuestra vista Batch tiene la foto exacta hasta ayer a las 20:15, hoy la tienda sigue abierta y están entrando nuevas compras cada segundo. Si esperáramos al proceso Batch de mañana, el negocio estaría "ciego" durante 24 horas.

Aquí entra la **Capa Speed**:
1. Utilizaremos **Spark Structured Streaming** para escuchar continuamente una carpeta de llegada de archivos JSON en vivo (`STREAM_INPUT_PATH`). En un entorno productivo real, esto suele conectarse a **Apache Kafka**, **AWS Kinesis** o **Azure Event Hubs**, pero usar archivos JSON en una carpeta de aterrizaje (*Landing Zone*) sigue exactamente el mismo código y funciona gratis en Databricks Community Edition sin servidores externos.
2. Calcularemos agregaciones en memoria en una tabla en vivo llamada `realtime_view`.

In [0]:
import json

# 1. Configuramos el lector de Streaming (readStream) vigilando la carpeta de entrada
df_ventas_stream = spark.readStream \
    .schema(esquema_ventas) \
    .option("maxFilesPerTrigger", 1) \
    .json(STREAM_INPUT_PATH) \
    .withColumn("fecha_hora", F.to_timestamp("fecha_hora"))

# 2. Definimos la agregación en tiempo real (Realtime View)
df_realtime_view = df_ventas_stream.groupBy("categoria").agg(
    F.count("id_transaccion").alias("total_transacciones_speed"),
    F.sum("monto").alias("ingresos_speed"),
    F.max("fecha_hora").alias("ultima_actualizacion_speed")
)

# 3. Iniciamos el stream guardando el resultado en una tabla en memoria ('vista_tiempo_real')
# Nota para Databricks Free / Colab: El sink 'memory' es ideal para laboratorios interactivos
query_stream = df_realtime_view.writeStream \
    .format("memory") \
    .queryName("vista_tiempo_real") \
    .outputMode("complete") \
    .start()

print("🚀 Motor de Streaming iniciado. Escuchando nuevos eventos en vivo...")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6881947136662331>, line 23
     11 df_realtime_view = df_ventas_stream.groupBy("categoria").agg(
     12     F.count("id_transaccion").alias("total_transacciones_speed"),
     13     F.sum("monto").alias("ingresos_speed"),
     14     F.max("fecha_hora").alias("ultima_actualizacion_speed")
     15 )
     17 # 3. Iniciamos el stream guardando el resultado en una tabla en memoria ('vista_tiempo_real')
     18 # Nota para Databricks Free / Colab: El sink 'memory' es ideal para laboratorios interactivos
     19 query_stream = df_realtime_view.writeStream \
     20     .format("memory") \
     21     .queryName("vista_tiempo_real") \
     22     .outputMode("complete") \
---> 23     .start()
     25 print("🚀 Motor de Streaming iniciado. Escuchando nuevos eventos en vivo...")

File <command-6881947136662331>, line 11
      4 df_

### Simulando la llegada de compras en vivo (Micro-batches)
Ahora que nuestro motor de streaming está encendido escuchando, vamos a simular que los clientes hacen compras en este momento enviando dos lotes de eventos JSON a la carpeta de entrada.

In [0]:
import json

def procesar_stream():
    df_realtime_view.writeStream \
        .format("memory") \
        .queryName("vista_tiempo_real") \
        .outputMode("complete") \
        .option("checkpointLocation", STREAM_CHECKPOINT_PATH) \
        .trigger(availableNow=True) \
        .start() \
        .awaitTermination()

# Lote 1 de eventos en vivo (Compras de la mañana de hoy)
eventos_vivo_lote1 = [
    {"id_transaccion": "TX_101", "categoria": "Laptops", "monto": 1350.00, "fecha_hora": "2026-05-02 09:05:00"},
    {"id_transaccion": "TX_102", "categoria": "Audio", "monto": 300.00, "fecha_hora": "2026-05-02 09:06:15"},
    {"id_transaccion": "TX_103", "categoria": "Audio", "monto": 120.00, "fecha_hora": "2026-05-02 09:08:40"}
]

# Escribimos el archivo JSON en la carpeta vigilada por el Stream
with open(f"{STREAM_INPUT_PATH}/eventos_1.json", "w") as f:
    for evento in eventos_vivo_lote1:
        f.write(json.dumps(evento) + "\n")

# Procesamos inmediatamente los datos disponibles en el stream
procesar_stream()

# Lote 2 de eventos en vivo (Compras unos minutos después, incluye una nueva categoría: 'Gaming')
eventos_vivo_lote2 = [
    {"id_transaccion": "TX_104", "categoria": "Smartphones", "monto": 900.00, "fecha_hora": "2026-05-02 09:15:00"},
    {"id_transaccion": "TX_105", "categoria": "Gaming", "monto": 600.00, "fecha_hora": "2026-05-02 09:18:22"}
]

with open(f"{STREAM_INPUT_PATH}/eventos_2.json", "w") as f:
    for evento in eventos_vivo_lote2:
        f.write(json.dumps(evento) + "\n")

# Procesamos el segundo lote
procesar_stream()

print("⚡ Vista en Tiempo Real (Realtime View) actualizada al instante por la Capa Speed:")
spark.sql("SELECT * FROM vista_tiempo_real ORDER BY ingresos_speed DESC").show(truncate=False)

⚡ Vista en Tiempo Real (Realtime View) actualizada al instante por la Capa Speed:
+-----------+-------------------------+--------------+--------------------------+
|categoria  |total_transacciones_speed|ingresos_speed|ultima_actualizacion_speed|
+-----------+-------------------------+--------------+--------------------------+
|Laptops    |1                        |1350.0        |2026-05-02 09:05:00       |
|Smartphones|1                        |900.0         |2026-05-02 09:15:00       |
|Gaming     |1                        |600.0         |2026-05-02 09:18:22       |
|Audio      |2                        |420.0         |2026-05-02 09:08:40       |
+-----------+-------------------------+--------------+--------------------------+



---
## Paso 3: Capa Serving (Unificación de Batch + Speed)

### Teoría rápida:
Observa lo que tenemos hasta ahora:
* En la **Vista Batch** (`df_batch_view`), tenemos categorías como `Monitores` (que tuvo ventas ayer, pero ninguna hoy) y ventas acumuladas de ayer.
* En la **Vista Speed** (`vista_tiempo_real`), tenemos ventas de hoy e incluso una categoría nueva (`Gaming`) que ayer no existía.

La **Capa Serving** se encarga de hacer un `FULL OUTER JOIN` (o `UNION ALL` seguido de una agregación) entre la **Vista Batch** y la **Vista Speed** para entregar al usuario una tabla única, transparente y completamente actualizada.

In [0]:
def consultar_capa_serving():
    # 1. Leemos la vista precalculada de la Capa Batch
    batch_df = spark.read.parquet(BATCH_VIEW_PATH)
    
    # 2. Leemos la vista en memoria de la Capa Speed
    speed_df = spark.table("vista_tiempo_real")
    
    # 3. Unimos ambas vistas mediante un FULL OUTER JOIN por 'categoria'
    # Usamos COALESCE para manejar los valores nulos si una categoría solo existe en Batch o solo en Speed
    df_serving = batch_df.join(speed_df, on="categoria", how="full_outer") \
        .select(
            F.col("categoria"),
            F.coalesce(F.col("total_transacciones_batch"), F.lit(0)).alias("tx_historico_batch"),
            F.coalesce(F.col("total_transacciones_speed"), F.lit(0)).alias("tx_hoy_speed"),
            (F.coalesce(F.col("total_transacciones_batch"), F.lit(0)) + 
             F.coalesce(F.col("total_transacciones_speed"), F.lit(0))).alias("total_transacciones_global"),
            F.coalesce(F.col("ingresos_batch"), F.lit(0.0)).alias("ingresos_batch"),
            F.coalesce(F.col("ingresos_speed"), F.lit(0.0)).alias("ingresos_speed"),
            (F.coalesce(F.col("ingresos_batch"), F.lit(0.0)) + 
             F.coalesce(F.col("ingresos_speed"), F.lit(0.0))).alias("ingresos_totales_combinados"),
            F.greatest(
                F.col("ultima_actualizacion_batch"), 
                F.col("ultima_actualizacion_speed")
            ).alias("ultimo_evento_registrado")
        ) \
        .orderBy(F.desc("ingresos_totales_combinados"))
    
    return df_serving

print("🎯 RESULTADO DE LA CAPA SERVING (Histórico Batch + Tiempo Real Speed):")
df_resultado_final = consultar_capa_serving()
df_resultado_final.show(truncate=False)

🎯 RESULTADO DE LA CAPA SERVING (Histórico Batch + Tiempo Real Speed):
+-----------+------------------+------------+--------------------------+--------------+--------------+---------------------------+------------------------+
|categoria  |tx_historico_batch|tx_hoy_speed|total_transacciones_global|ingresos_batch|ingresos_speed|ingresos_totales_combinados|ultimo_evento_registrado|
+-----------+------------------+------------+--------------------------+--------------+--------------+---------------------------+------------------------+
|Laptops    |3                 |1           |4                         |3800.0        |1350.0        |5150.0                     |2026-05-02 09:05:00     |
|Smartphones|3                 |1           |4                         |2500.0        |900.0         |3400.0                     |2026-05-02 09:15:00     |
|Audio      |2                 |2           |4                         |350.0         |420.0         |770.0                      |2026-05-02 09:08:40 

---
## Paso 4: El Cierre del Ciclo (Sincronización Batch Nocturna)

### ¿Qué ocurre al final del día en una Arquitectura Lambda?
Los datos que hoy entraron por la **Capa Speed** también se guardan en el **Master Dataset** de la **Capa Batch**. Cuando llega la noche:
1. El proceso **Batch** vuelve a ejecutarse incluyendo los datos nuevos.
2. La **Vista Speed** se reinicia (se limpia), ya que esos datos ahora forman parte de la **Vista Batch** consolidada.

Veamos cómo funciona esta sincronización en código y detengamos nuestro stream de forma limpia.

In [0]:
# 1. Al final del día, los eventos JSON recibidos hoy se consolidan en el Master Dataset (Capa Batch)
df_nuevos_para_batch = spark.read.schema(esquema_ventas).json(STREAM_INPUT_PATH) \
    .withColumn("fecha_hora", F.to_timestamp("fecha_hora"))

df_nuevos_para_batch.write.mode("append").parquet(RAW_BATCH_PATH)

# 2. Re-ejecutamos el Job Batch leyendo TODO el Master Dataset actualizado
df_master_actualizado = spark.read.parquet(RAW_BATCH_PATH)

df_nueva_batch_view = df_master_actualizado.groupBy("categoria").agg(
    F.count("id_transaccion").alias("total_transacciones_batch"),
    F.sum("monto").alias("ingresos_batch"),
    F.max("fecha_hora").alias("ultima_actualizacion_batch")
)

df_nueva_batch_view.write.mode("overwrite").parquet(BATCH_VIEW_PATH)

print("🔄 Nueva Vista Batch recalculada tras el cierre del día (contiene todo el historial + hoy):")
df_nueva_batch_view.orderBy(F.desc("ingresos_batch")).show(truncate=False)

# 3. Detenemos el motor de streaming para liberar recursos del clúster
query_stream.stop()
print("🛑 Stream detenido correctamente. Recursos liberados.")

🔄 Nueva Vista Batch recalculada tras el cierre del día (contiene todo el historial + hoy):
+-----------+-------------------------+--------------+--------------------------+
|categoria  |total_transacciones_batch|ingresos_batch|ultima_actualizacion_batch|
+-----------+-------------------------+--------------+--------------------------+
|Laptops    |4                        |5150.0        |2026-05-02 09:05:00       |
|Smartphones|4                        |3400.0        |2026-05-02 09:15:00       |
|Audio      |4                        |770.0         |2026-05-02 09:08:40       |
|Monitores  |2                        |750.0         |2026-05-01 18:20:00       |
|Gaming     |1                        |600.0         |2026-05-02 09:18:22       |
+-----------+-------------------------+--------------+--------------------------+



---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6881947136662337>, line 22
     19 df_nueva_batch_view.orderBy(F.desc("ingresos_batch")).show(truncate=False)
     21 # 3. Detenemos el motor de streaming para liberar recursos del clúster
---> 22 query_stream.stop()
     23 print("🛑 Stream detenido correctamente. Recursos liberados.")

NameError: name 'query_stream' is not defined

---
## 5. Conclusiones y Reflexión de Ingeniería de Datos


1. **Inmutabilidad y Precisión (Capa Batch):** Guardaste los datos crudos en formato columnar `Parquet` y generaste vistas históricas precalculadas.
2. **Baja Latencia (Capa Speed):** Utilizaste `Spark Structured Streaming` para procesar eventos JSON al vuelo sin esperar al proceso nocturno.
3. **Vista Unificada (Capa Serving):** Combinaste ambas fuentes con `Full Outer Join` y `Coalesce` para que el negocio nunca vea datos incompletos ni desactualizados.

### 💡 Nota profesional: De Lambda a la Arquitectura Medallion (Delta Lake)
Como habrás notado, en la Arquitectura Lambda tuvimos que escribir la lógica de agregación (`groupBy`, `count`, `sum`) **dos veces**: una para el código Batch y otra para el código Streaming. Mantener dos bases de código distintas en producción puede ser costoso.

Por esa razón, en plataformas modernas como **Databricks**, la Arquitectura Lambda evolucionó hacia la **Arquitectura Lakehouse / Medallion (Bronce -> Plata -> Oro)** usando **Delta Lake**, que permite unificar tablas Batch y Streaming en una sola capa transaccional. ¡Dominar este ejemplo de Arquitectura Lambda te da las bases fundamentales para entender cómo y por qué funcionan los sistemas modernos de Big Data!